# Projet - Deep Learning

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
import ipywidgets as widgets

## Contexte

Le diabète fait partie des maladies chroniques les plus courantes aux États-Unis et touche chaque année des millions de personnes. Cette affection sérieuse se traduit par une mauvaise régulation du sucre dans le sang, ce qui peut, à long terme, réduire la qualité de vie et même l’espérance de vie. Au moment de la digestion, les aliments se transforment en sucres qui passent dans le sang. Cela incite le pancréas à produire de l’insuline, une hormone qui permet aux cellules d’utiliser ces sucres comme source d’énergie. Dans le cas du diabète, soit l’organisme ne produit pas assez d’insuline, soit il réagit mal.

L’excès prolongé de sucre dans le sang augmente les risques de complications graves : maladies cardiovasculaires, perte de la vue, amputations ou encore insuffisance rénale. Même si le diabète ne se guérit pas, il est possible d’en limiter les effets grâce à la perte de poids, à une alimentation adaptée, à une activité physique régulière et à un suivi médical approprié. Plus le diagnostic est posé tôt, plus il est possible d’agir efficacement. C’est pour cette raison que les outils de prédiction du risque de diabète représentent un enjeu important de santé publique.

L’ampleur du phénomène est frappante. Selon les Centres pour le contrôle et la prévention des maladies (CDC) en 2018, 34,2 millions d’Américains vivaient avec un diabète, et 88 millions étaient prédiabétiques. Une personne diabétique sur cinq n’en avait pas conscience, et la grande majorité des personnes prédiabétiques ignorait également leur situation. Le diabète de type 2, la forme la plus fréquente, est influencé par de nombreux facteurs : âge, niveau d’études, revenus, zone géographique, origine ethnique ou encore conditions sociales de vie. Les populations les plus défavorisées sont souvent les plus touchées.

La maladie représente aussi un poids financier immense : le coût du diabète diagnostiqué est estimé à environ 327 milliards de dollars par an, et si l’on inclut le diabète non diagnostiqué et le prédiabète, on approche les 400 milliards de dollars.

**Objectifs :**

Nous sommes chargés de construire un modèle de classification binaire permettant de prédire la probabilité qu’une personne soit diabétique à partir des données médicales issues de l’enquête diabetes_binary_health_indicators_BRFSS201 (BRFSS 2015).

Notre objectif ultime est de maximiser la métrique ROC AUC sur un jeu de test non étiqueté.
En parallèle, nous adopterons une approche MLOps (versioning des données, traçabilité et automatisation des traitements) et nous travaillerons en mode Scrum, selon trois sprints, livrables à chaque itération.

## Présentation du jeu de données

Le Système de surveillance des facteurs de risque comportementaux (BRFSS) est une enquête téléphonique annuelle menée par les Centres pour le contrôle et la prévention des maladies (CDC) sur la santé. Chaque année, cette enquête recueille les réponses de plus de 400 000 Américains sur leurs comportements à risque pour la santé, leurs maladies chroniques et leur recours aux services de prévention. Elle est réalisée chaque année depuis 1984. Pour ce projet, nous avons utilisé un fichier CSV des données disponibles sur [Kaggle](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset) pour l'année 2015. Ce jeu de données contient les réponses de 253 680 personnes et comporte 22 variables. Ces variables correspondent soit à des questions posées directement aux participants, soit à des variables calculées à partir de leurs réponses individuelles.

Les variables du dataset sont :
- **Diabetes_binary :** Objectif de l’étude : indique si la personne est diabétique (1 = Oui, 0 = Non).

- **HighBP :** Indique si le sujet souffre d’hypertension artérielle. 
- **HighChol :** Le sujet a un taux de cholestérol élevé. 
- **CholCheck :** Le sujet a effectué un contrôle du cholestérol dans les 5 dernières années. 
- **BMI :** Indice de masse corporelle. 
- **Smoker :** Le sujet a fumé plus de 100 cigarettes au cours de sa vie. 
- **Stroke :** Le sujet a déjà eu un AVC. 
- **HeartDiseaseorAttack :** Le sujet a déjà eu une maladie coronarienne ou un infarctus du myocarde. 
- **PhysActivity :** Le sujet a pratiqué une activité physique au cours des 30 derniers jours (hors travail). 
- **Fruits :** Le sujet consomme des fruits au moins une fois par jour. 
- **Veggies :** Le sujet consomme des légumes au moins une fois par jour. 
- **HvyAlcoholConsump :** Forte consommation d’alcool (hommes ≥ 14 verres/semaine, femmes ≥ 7). 
- **AnyHealthcare :** Le sujet bénéficie d’une couverture santé (assurance ou mutuelle). 
- **NoDocbcCost :** Le sujet n’a pas consulté un médecin dans les 12 derniers mois à cause du coût. 
- **GenHlth :** État de santé général (1 = excellent, 2 = très bon, 3 = bon, 4 = moyen, 5 = mauvais). 
- **MentHlth :** Nombre de jours de mauvaise santé mentale sur les 30 derniers jours. 
- **PhysHlth :** Nombre de jours de maladie ou blessure sur les 30 derniers jours. 
- **DiffWalk :** Difficulté à marcher ou monter des escaliers. 
- **Sex :** Sexe du sujet (0 = Femme, 1 = Homme). 
- **Age :** Tranche d’âge (1 = 18–24, …, 13 = 80+). 
- **Education :** Niveau d’éducation (1 = jamais scolarisé, …). 
- **Income :** Niveau de revenu (1 < 10k$, …, 8 < 75k$).


### Séparation des données

Il est primordial de séparer les données en deux catégories bien distinctes :
- Variable cible : elle représente le résultat que l’on cherche à prédire ou à expliquer à l’aide du modèle.

- Variables explicatives : elles correspondent aux variables utilisées pour prédire, expliquer ou décrire la variable cible.

Dans ce contexte, la seule variable cible, c'est-à-dire celle que nous allons de voir prévoir dans notre projet sera **Diabetes_binary**.

Tout le reste des variables sont des variables explicatives. 

On distingue principalement deux types de variables : les variables quantitatives et les variables qualitatives. Cette distinction est essentielle pour choisir les méthodes d’analyse et de modélisation adaptées.

Une variable quantitative (ou numérique) correspond à une information mesurable qui s’exprime par un nombre (ex: Age, poids, température, supercifie, ...). Dans notre contexte les variables sont considérés comme quantitative continue (variable qui prend un infini de valeurs réelles à l'intérieur d'un interval donné) :
- BMI
- MenHlth
- PhysHlth

Une variable qualitative (ou catégorielle) décrit une caractéristique non numérique. Elle représente des catégories ou des états. Il y a 2 types de variables qualitatives : qualitative nominale (étiquetage, binaire) et qualitative ordinale (données définies dans une relation d'ordre entre différentes catégories possibles). Variables qualitatives nominales :
- Diabetes_binary
- HighBP
- HighChol
- CholCheck
- Smoker
- Stroke
- HeartDiseaseorAttack 
- PhysActivity 
- Fruits
- Veggies 
- HvyAlcoholConsump 
- AnyHealthcare
- NoDocbcCost
- DiffWalk
- Sex 

Variables ordinales :
- GenHlth
- Age
- Income
- Education

### Variables sensibles 

À des fins d’anonymisation et de limitation des biais, nous avons choisi de retirer certaines variables non essentielles à la prédiction du diabète.
- **Income :** le niveau de revenu peut être indirectement lié au diabète (accès aux soins, alimentation), mais nous choisissons de l’exclure afin d’éviter d’introduire des biais socio-économiques dans le modèle.
- **Education :** de même, le niveau d’éducation peut influencer certains comportements de santé, mais il n’est pas un facteur médical direct. Nous le retirons afin de limiter les biais et de conserver un modèle centré sur des variables de santé.

Le dataset contient également certaines variables pouvant être considérées comme subjectives. En effet, ces variables reposent sur des auto-évaluations des individus, ce qui peut introduire des biais liés à la perception personnelle ou à l’état mental.
- **GenHlth**
- **MenHlth**
- **PhysHlth**

### Scissions des données (Train/Test/Validation)

Afin d'entrainer au mieux notre modèle avec nos donnéees, nous allons séparer notre dataset en 3 sous-dataset.

- **Dataset d'entrainement :** Ce jeu de données est utilisé pour entraîner le modèle. Il permet au modèle d’apprendre les relations et les patterns présents dans les données.
- **Dataset de validation :** Ce jeu de données est utilisé pendant l’entraînement pour évaluer les performances du modèle de manière intermédiaire. Il permet d’ajuster les hyperparamètres et de choisir le meilleur modèle, tout en limitant le surapprentissage (overfitting).
- **Dataset de test :** Ce jeu de données est utilisé uniquement à la fin du processus. Il permet d’évaluer les performances finales du modèle sur des données qu’il n’a jamais vues. On compare alors les valeurs prédites aux valeurs réelles afin d’obtenir une estimation fiable de sa capacité de généralisation.

Afin d'avoir une bonne répartition de nos données, nous allons utiliser un découpage de 70% des données dans le dataset d'entrainement, 15% de validation et 15% de donnéees de test. 

## Pré-traitement des données

### Loading du dataset

In [13]:
path = "Health_Data/diabetes_binary_health_indicators_BRFSS2015.csv"

In [14]:
def load_dataset(path):
    dataset = pd.read_csv(path)
    return dataset

Suppression des variables sensibles :

In [32]:
def remove_sensible_data(df, columns_to_remove):
    df = df.copy()
    df = df.drop(columns=columns_to_remove)
    return df

### Gestion des valeurs manquantes

Les valeurs manquantes sont un problème courant dans les ensembles de données réels et peuvent nuire aux performances des modèles de Machine Learning. Il existe plusieurs méthodes pour gérer les valeurs manquantes dans un dataset :
- **Suppression de la ligne :** supprimer les observations contenant des valeurs manquantes peut être pertinent, mais si ces valeurs sont nombreuses, cela peut entraîner une perte importante d’information.
- **Remplir avec la médiane :** méthode robuste, particulièrement adaptée aux variables numériques contenant des valeurs extrêmes (outliers).
- **Remplir avec la moyenne :** adaptée aux variables numériques sans forte présence d’outliers.
- **Remplir avec la valeur suivante (forward fill):** consiste à remplacer la valeur manquante par la valeur suivante dans la série.
- **Remplir avec la valeur précédente (backward fill):** consiste à remplacer la valeur manquante par la valeur précédente.
- **Remplir avec le mode :** remplace la valeur manquante par la valeur la plus fréquente, généralement utilisée pour les variables catégorielles.

Pour toutes nos valeurs, nous avons choisi de remplir les valeurs manquantes par le "mode", c'est à dire la valeur la plus fréquente car toutes nos valeurs sont catégorielles.

In [ ]:
def check_missing_values(df):
    missing_values_columns = []
    for i in range(len(df.columns)):
        if df.iloc[:, i].isnull().sum() > 0:
            missing_values_columns.append(df.columns[i])
            print(f"Column {df.columns[i]} has {df.iloc[:, i].isnull().sum()} missing values.")
    
    if not missing_values_columns:
        print("No missing values found in the dataset.")
    return missing_values_columns

def fill_missing_columns(df, missing_columns):
    # Fill with mode
    df[missing_columns] = df[missing_columns].fillna(df[missing_columns].mode().iloc[0])
    print(f"Column {missing_columns} filled with value: {df[missing_columns].mode().iloc[0]}")

    return df

### Management des valeurs aberrantes 

In [43]:
def outliers_detection(df):

    df = df.copy()

    z_scores = df.apply(zscore)

    for col in df.columns:
        outliers = z_scores[col].abs() > 3
        if outliers.any():
            df[col] = df[col].mask(outliers, df[col].median())
            print(f"Outliers detected in column {col} and replaced with mode value.")
    
    return df

### Normalisation des données

In [ ]:
# def normalize_data(df):
#     df = df.copy()
#     scaler = MinMaxScaler()
#     df = scaler.fit_transform(df)
#     df = pd.DataFrame(df, columns=df_dia.columns)
#     return df

### Traitement (main)

In [44]:
df_data = load_dataset(path)
df_data = remove_sensible_data(df_data, ['GenHlth', 'MentHlth', 'PhysHlth', 'Income', 'Education'])

columns_missing = check_missing_values(df_data)
df_data = fill_missing_columns(df_data, columns_missing)

# Gestion des doublons

df_data = outliers_detection(df_data)

# Normalisation des données 

# Analyse Exploratoire quantitative + qualitative

# Save du dataset nettoyé

No missing values found in the dataset.
Column [] filled with value: Series([], dtype: float64)
Outliers detected in column CholCheck and replaced with mode value.
Outliers detected in column BMI and replaced with mode value.
Outliers detected in column Stroke and replaced with mode value.
Outliers detected in column HeartDiseaseorAttack and replaced with mode value.
Outliers detected in column HvyAlcoholConsump and replaced with mode value.
Outliers detected in column AnyHealthcare and replaced with mode value.
Outliers detected in column NoDocbcCost and replaced with mode value.


In [46]:
df_data.describe()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,DiffWalk,Sex,Age
count,253680.000000,253680.000000,253680.000000,253680.0,253680.000000,253680.000000,253680.0,253680.0,253680.000000,253680.000000,253680.000000,253680.0,253680.0,253680.0,253680.000000,253680.000000,253680.000000
mean,0.139333,0.429001,0.424121,1.0,28.010679,0.443169,0.0,0.0,0.756544,0.634256,0.811420,0.0,1.0,0.0,0.168224,0.440342,8.032119
std,0.346294,0.494934,0.494210,0.0,5.576425,0.496761,0.0,0.0,0.429169,0.481639,0.391175,0.0,0.0,0.0,0.374066,0.496429,3.054220
min,0.000000,0.000000,0.000000,1.0,12.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,1.0,24.000000,0.000000,0.0,0.0,1.000000,0.000000,1.000000,0.0,1.0,0.0,0.000000,0.000000,6.000000
50%,0.000000,0.000000,0.000000,1.0,27.000000,0.000000,0.0,0.0,1.000000,1.000000,1.000000,0.0,1.0,0.0,0.000000,0.000000,8.000000
75%,0.000000,1.000000,1.000000,1.0,31.000000,1.000000,0.0,0.0,1.000000,1.000000,1.000000,0.0,1.0,0.0,0.000000,1.000000,10.000000
max,1.000000,1.000000,1.000000,1.0,48.000000,1.000000,0.0,0.0,1.000000,1.000000,1.000000,0.0,1.0,0.0,1.000000,1.000000,13.000000
